# Make predictors

This section computes the acoustic predictor variables used as regressors in the TRF models. The pipeline runs in two steps:

1. **Gammatone spectrograms** — each stimulus audio file (`.wav` / `.mp3`) is passed through a gammatone filter bank (80–15 000 Hz, 128 channels, 1 ms time step) using `eelbrain.gammatone_bank`. The resulting high-resolution spectrograms are cached as `.pickle` files in `stimuli/`.

2. **Predictor derivatives** — from each spectrogram a set of standard predictor variants is derived and saved to `predictors/`:
   - `~gammatone-1` — log-compressed temporal envelope (summed across all frequency channels → 1 band)
   - `~gammatone-on-1` — acoustic onset envelope (edge-detector applied to the log spectrogram, 1 band)
   - `~gammatone-8` / `~gammatone-on-8` — log envelope / onset binned into 8 frequency bands
   - `~gammatone-lin-8` / `~gammatone-pow-8` — linear-scale and power-law ($x^{0.6}$) spectrograms at 8 bands
   - `~gammatone-onDer-1` — half-wave rectified temporal derivative (1 band)

Both steps skip files that have already been processed, so the cells are safe to re-run.

In [ ]:
from pathlib import Path
import eelbrain as eel
import numpy as np
from pydub import AudioSegment
from helpers import make_gammatone, make_gt_predictor

ROOT = Path.cwd()
STIMULUS_DIR = ROOT / 'stimuli'
PREDICTOR_DIR = ROOT / 'predictors'
PREDICTOR_DIR.mkdir(exist_ok=True)

## Make gammatone files

Generates high-resolution gammatone spectrograms from stimulus audio files using `eelbrain.gammatone_bank` (80–15000 Hz, 128 channels, 1 ms time step) and saves them as `.pickle` files. Adapted from [`make_gammatone.py`](https://github.com/Eelbrain/Alice/blob/main/predictors/make_gammatone.py) in the Alice dataset.

In [ ]:
filenames = [f.name for f in STIMULUS_DIR.iterdir() if f.is_file()]
for ii in filenames: 
    i = ii[:-4]
    filepath_wav = STIMULUS_DIR / f'{i}.wav'
    filepath_mp3 = STIMULUS_DIR / f'{i}.mp3'
    
    if filepath_wav.exists() and filepath_wav.stat().st_size > 0:
        print(f"Read {filepath_wav}")
        wav = eel.load.wav(filepath_wav)
    elif filepath_mp3.exists():
        print(f"Read {filepath_mp3}")

        audio = AudioSegment.from_mp3(filepath_mp3)
        samples = np.array(audio.get_array_of_samples())
        samples = samples.astype(np.float32) / (2**15)  # normalize 16-bit
        fs = audio.frame_rate
        
        # convert to eelbrain NDVar
        time = eel.UTS(0, 1/fs, samples.shape[0])
        wav = eel.NDVar(samples, (time,))
    
    else:
        print(f"File {i} is missing or empty...")
        continue
    
    make_gammatone(wav,i)

## Make gammatone predictors

Derives predictor variables from the gammatone spectrograms created above. Applies log scaling and an onset filter, then sums across frequency bands to produce 1-band and 8-band envelope predictors saved as `.pickle` files. Adapted from [`make_gammatone_predictors.py`](https://github.com/Eelbrain/Alice/blob/main/predictors/make_gammatone_predictors.py) in the Alice dataset.

In [ ]:
filenames = [f.name for f in STIMULUS_DIR.iterdir() if f.is_file() and (f.name.endswith('.pickle'))]
name = '-gammatone'

conv_names = ['left', 'right', 'mean']

nfiles = len(filenames)

for nn, ii in enumerate(filenames):
    print(f'file: {nn+1}/{nfiles}   {filenames[nn]}')

    i = ii[:-7]
    j = i[:-len(name)]
    dst = PREDICTOR_DIR / f'{j}~gammatone-1.pickle'
    if dst.exists():
       continue
    make_gt_predictor(i,j)

# Preprocess EEG

This section loads the raw EEG recordings and prepares them for TRF analysis. The pipeline covers five steps that must be run in order:

1. **Convert XDF → FIF** — reads each raw `.xdf` recording (one per subject and sensor array) and saves it as an MNE `.fif` file under `data/sub-<id>/eeg/`. This format is required by the eelbrain TRF pipeline. Run once per subject.

2. **Initiate TRF pipeline** — imports libraries, resolves data paths, patches the MNE-BIDS key validator to allow compound acquisition names (e.g. `scalpSustA`), and discovers all subject folders under `data/`.

3. **Make ICA** — fits an ICA decomposition (20 components for scalp, 10 for cEEGrid) on the 1–40 Hz band-pass filtered data and saves the solution to `data/derivatives/ica/`. Run once per subject.

4. **Select ICA components to exclude** — opens an interactive GUI to manually inspect and mark artifactual ICA components (eye blinks, muscle noise, cardiac, line noise) for removal. Run once per subject after ICA fitting.

5. **Align EEG and predictors** — applies the accepted ICA solution to the 0.1–40 Hz filtered data, re-filters to 1–20 Hz, extracts stimulus-locked epochs, resamples to 50 Hz, z-scores each channel, and time-aligns the corresponding gammatone predictor variables. For the switA condition, epochs and predictors are additionally split at the attention-switch boundaries (t = 35 s and t = 125 s). The final EEG + predictor bundles are saved as `.pickle` files under `data/derivatives/preprocessed/`.

## Make .xdf to .fif files

The cell:
1. Converts the raw `.xdf` recording to MNE `.fif` format (via `save_xdf_as_fif`) if not already done. The correct data structure need a .fif file to initiate eelbrain TRF pipeline.

In [ ]:
from eelbrain import *
from helpers import save_xdf_as_fif
from pathlib import Path
import re

ROOT = Path.cwd()
DATA_ROOT = Path.cwd() / 'data'

subjects = [
    p.name.removeprefix("sub-")
    for p in DATA_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("sub-")
]
print(f'Subjects: {subjects}')

exclude_subs = []
filt = [1, 20]
aquisitions = ['scalpSustA', 'ceegridSustA', 'scalpSwitA', 'ceegridSwitA', 'scalpConvA', 'ceegridConvA']

for subject in subjects:
    print(f'sub-{subject}')
    for acq_combined in aquisitions:
        acq_type = re.match(r'^(scalp|ceegrid)', acq_combined).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq_combined)
        task = task_cap[0].lower() + task_cap[1:]

        dst1 = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_task-{task}_eeg.xdf'
        if not dst1.exists():
            print(f'  {acq_combined}: xdf file not found, skipping')
            continue

        # No-task file saved by save_xdf_as_fif (e.g. sub-99_acq-scalpSustA_eeg.fif)
        dst2 = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_acq-{acq_combined}_eeg.fif'
        if not dst2.exists():
            print(f'  {acq_combined}: making fif files...')
            save_xdf_as_fif(task, subject)

## Initiate TRF pipeline

The TRF pipline can only be initiated correctly if:

        DATA_ROOT / f'sub-{id}' / 'eeg' / 'sub-{id}_acq-{acq}_eeg.fif'
        
files exist (for example: sub-99_acq-scalpSustA_eeg.fif). To make these .fif files from raw .xdf files, run previous cell. 

NOTE (experiment.py):

    tasks = tasks = ["sustA"] #, "switA", "convA"]

since we only provide data for sustA.

In [ ]:
from eelbrain import *
from experiment import mobEEG_e, PARAMETERS 
from helpers import save_xdf_as_fif, make_gammatone, make_gt_predictor
import eelbrain as eel
import numpy as np
from scipy.signal import resample
import shutil


# Allow underscores in BIDS 'acq' field (e.g. 'sustA_scalp') — eelbrain passes
# the acquisition value through to mne_bids which otherwise rejects underscores.
import mne_bids.utils as _mbu
_orig_check = _mbu._check_key_val
def _patched_check(key, val):
    if key == 'acq':
        return key, val
    return _orig_check(key, val)
_mbu._check_key_val = _patched_check

SAVE_DIR = ROOT / 'results'
SAVE_DIR.mkdir(exist_ok=True)


## Make ICA

Fits an Independent Component Analysis (ICA) decomposition for each subject and sensor array (scalp / cEEGrid) across all three task conditions (sustA, switA, convA). **Only needs to be run once per subject.**

The cell:
1. Creates a BIDS-compatible copy of the `.fif` file with the `task-` field in the filename, as required by eelbrain's pipeline.
2. Calls `mobEEG_e.make_ica()` which band-pass filters the data (1–40 Hz) and fits an ICA solution — 20 components for scalp, 10 for cEEGrid — saving the result to `data/derivatives/ica/`.

Subjects or acquisitions for which an ICA file already exists are silently skipped.

In [ ]:
ica_dir = DATA_ROOT / 'derivatives' / 'ica'
ica_dir.mkdir(parents=True, exist_ok=True)

for subject in subjects:
    print(f'sub-{subject}')
    for acq_combined in aquisitions:
        acq_type = re.match(r'^(scalp|ceegrid)', acq_combined).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq_combined)
        task = task_cap[0].lower() + task_cap[1:]

        dst1 = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_task-{task}_eeg.xdf'
        if not dst1.exists():
            print(f'  {acq_combined}: xdf file not found, skipping')
            continue

        # No-task file saved by save_xdf_as_fif (e.g. sub-99_acq-scalpSustA_eeg.fif)
        dst2 = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_acq-{acq_combined}_eeg.fif'
        if not dst2.exists():
            print(f'  {dst2} is missing... Run cell "make .xdf to .fif"')
            continue

        # Eelbrain's BIDS path builder requires task- in the filename; create a copy
        dst2_bids = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_task-{task}_acq-{acq_combined}_eeg.fif'
        if not dst2_bids.exists():
            shutil.copy2(dst2, dst2_bids)

        ica_file = ica_dir / f'sub-{subject}_acq-{acq_combined}_eeg_raw-ica_ica.fif'
        if ica_file.exists():
            print(f'  {acq_combined}: ICA already exists, skipping')
            continue

        try:
            mobEEG_e.set(subject=subject, raw=f'{task}_ica_{acq_type}', acquisition=acq_combined)
            mobEEG_e.make_ica()
            print(f'  Saved: {ica_file.name}')
        except FileNotFoundError as err:

            print(f'  {acq_combined}: file not found — {err}')

## Select ICA components to exclude

Opens the eelbrain ICA selection GUI for each subject and sensor array so that artifactual components (eye blinks, muscle noise, cardiac, line noise) can be manually identified and marked for removal. **Only needs to be run once per subject**, after ICA has been fitted above.

The cell iterates over all subjects and acquisitions and calls `mobEEG_e.make_ica_selection()`, which renders each component as a time-series, scalp topography, and power spectrum. Click components to toggle their exclusion status and close the window to save the selection.

> **Note:** The GUI runs inside the notebook kernel's event loop. If the window freezes or does not respond, use `select_ica_gui.py` instead, which launches the GUI in a separate process:
> ```python
> import subprocess, sys
> subprocess.run([sys.executable, 'select_ica_gui.py', subject, condition, acq_type])
> ```
> The selected components are stored back in the condition-specific ICA file under `data/derivatives/ica/` and applied automatically during the **Align EEG and predictors** step below.

In [ ]:
ica_dir = DATA_ROOT / 'derivatives' / 'ica'

for subject in subjects:
    print(f'sub-{subject}')
    for acq_combined in aquisitions:
        acq_type = re.match(r'^(scalp|ceegrid)', acq_combined).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq_combined)
        task = task_cap[0].lower() + task_cap[1:]

        ica_file = ica_dir / f'sub-{subject}_acq-{acq_combined}_eeg_raw-ica_ica.fif'
        if not ica_file.exists():
            print(f'  {acq_combined}: ICA not found, skipping')
            continue


        mobEEG_e.set(subject=subject, raw=f'{task}_ica_{acq_type}', acquisition=acq_combined)

        mobEEG_e.make_ica_selection()
        print(f'  {acq_combined}: done')

# Align EEG and predictors

In [ ]:
SAVE_DIR = DATA_ROOT / 'derivatives' / 'preprocessed'
SAVE_DIR.mkdir(exist_ok=True)


fieldnames = ['eeg','predictors','microphones']
attentions = ['attended','ignored','null']

# Combined condition_acquisition names
typeOfRegressors = ['~gammatone-1','~gammatone-on-1']
micRegressors = ['gt_log1','gt_on1']

predictor_names = ['env', 'onset']

data = {
        'eeg': [],
        'info': {},
        'microphones': {},
        'predictors': {
            attention: {
                pred: []
                for pred in predictor_names
            }
            for attention in attentions
        }
    }

info = {
    'summary': 'Preprocessed data ready for TRF analysis.',
    'condition': '',
    'acquisition': '',
    'attended': 'Attended speech',
    'ignored': 'Ignored speech',
    'null': 'Trial-shifted predictors as null model',
    'normalization': 'eeg, predictors and mics are z-normalized',
    'filter': filt,
    'fs': 50,
    'cut': 'First and last 1s of each block are cut to avoid edge artifacts from filtering and epoching',
}



info_switA = {
    'summary': 'Preprocessed data ready for TRF analysis. Switches are randomized between the intervals',
    'block1': [0,35],
    'switch1': [35,55],
    'block2': [55,125],
    'switch2': [125,145],
    'block3': [145,178],
    'attended': 'Predictors for attended block1 and block3',
    'ignored': 'Predictors for attended block2',
    'null': 'Trial-shifted predictors as null model',
}


for subject in subjects:
    print('---------------------------')
    print(f'Subject: {subject}')
    out_path = SAVE_DIR / f'sub-{subject}'
    out_path.mkdir(parents=True, exist_ok=True)

    for acq_combined in aquisitions:
        acq_type = re.match(r'^(scalp|ceegrid)', acq_combined).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq_combined)
        task = task_cap[0].lower() + task_cap[1:]
        print(f'Condition: {task}, Acq: {acq_combined}')

        info['condition'] = task
        info['acquisition'] = acq_combined

        if task == 'switA':
            info['switchingInfo'] = info_switA
        else:
            info['switchingInfo'] = 'No switching'

        eeg_fif = DATA_ROOT / f'sub-{subject}' / 'eeg' / f'sub-{subject}_acq-{acq_combined}_eeg.fif'
        if not eeg_fif.exists():
            print(f'  {acq_combined}: EEG file not found, skipping')
            continue

        save_path = out_path / f'sub-{subject}_acq-{acq_combined}.pickle'
        if save_path.exists():
            continue

        mobEEG_e.set(subject=subject, task=task, raw=f'1-20_{task}_{acq_type}', acquisition=acq_combined)

        epochs, predictors, predictors_mic, predictors_attended, attended_conv = mobEEG_e.align_epochs_and_predictors(
            typeOfRegressors, micRegressors, filt, condition=task
        )

        eeg_ndvar_all = eel.load.mne.epochs_ndvar(epochs)
        data['eeg'] = eeg_ndvar_all

        data['predictors']['attended']['env'] = predictors['attended_files']['~gammatone-1'][0]
        data['predictors']['attended']['onset'] = predictors['attended_files']['~gammatone-on-1'][0]
        data['predictors']['ignored']['env'] = predictors['ignored_files']['~gammatone-1'][0]
        data['predictors']['ignored']['onset'] = predictors['ignored_files']['~gammatone-on-1'][0]
        data['predictors']['null']['env'] = predictors['shifted_files']['~gammatone-1'][0]
        data['predictors']['null']['onset'] = predictors['shifted_files']['~gammatone-on-1'][0]

        data['info'] = info

        eel.save.pickle(data, save_path)

# Quick TRF-check

A lightweight sanity check to verify that the preprocessed data for a single subject looks reasonable before running the full group analysis. Loads one subject's `.pickle` file and fits both backward and forward TRF models using `eelbrain.boosting` with L1 error and 9-fold cross-validation.

- **Backward model** (predictor → EEG, lag window −0.6 – 0.2 s) — estimates how well the acoustic envelope predicts the EEG response. Reports the cross-validated correlation *r* for attended, ignored, and null (trial-shifted) predictors. Attended *r* should be clearly higher than null.
- **Forward model** (EEG → predictor, lag window −0.2 – 0.6 s) — estimates the temporal response function (TRF) as a scalp topography over time. The attended TRF should show clear auditory components (e.g. N1 at ~100 ms); the null TRF should be flat.

Set `acq` at the top of the load cell to switch between sensor arrays and conditions (e.g. `'scalpSustA'`, `'ceegridSwitA'`).

In [ ]:
acq = 'scalpSustA'

tmp = SAVE_DIR / f'sub-{subject}' /  f'sub-{subject}_acq-{acq}.pickle'
data = eel.load.unpickle(tmp)

eeg = data['eeg']
attended = data['predictors']['attended']['env']
ignored = data['predictors']['ignored']['env']
null = data['predictors']['null']['env']

## Backward models

In [ ]:
TMIN = -0.6
TMAX = 0.2
nPartitions = 9

bw_att = eel.boosting(attended, eeg, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)
print(f'Attended r: {float(bw_att.r)}')

bw_ign = eel.boosting(ignored, eeg, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)
print(f'Ignored r: {float(bw_ign.r)}')

bw_null = eel.boosting(null, eeg, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)
print(f'Null r: {float(bw_null.r)}')



## Forward models

In [ ]:
TMIN = -0.2
TMAX = 0.6

fw_att = eel.boosting(eeg, attended, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)

fw_ign = eel.boosting(eeg, ignored, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)

fw_null = eel.boosting(eeg, null, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)


vmax = float(max(
    abs(fw_att.h_scaled.x).max(),
    abs(fw_ign.h_scaled.x).max(),
    abs(fw_null.h_scaled.x).max()
))

def clean_spines(p):
    for ax in p.axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.axvline(0, color='grey', linestyle='--', alpha=0.6)
        ax.grid(True, alpha=0.2)

p_att = eel.plot.Butterfly(fw_att.h_scaled, title='Attended TRF', vmax=vmax)
clean_spines(p_att)

p_ign = eel.plot.Butterfly(fw_ign.h_scaled, title='Ignored TRF', vmax=vmax)
clean_spines(p_ign)

p_null = eel.plot.Butterfly(fw_null.h_scaled, title='Null TRF', vmax=vmax)
clean_spines(p_null)